In [6]:
# A default setup cell.
# It imports environment variables, define 'devtools.debug" as a buildins, set PYTHONPATH, and code auto-reload
# Copy it in other Notebooks

%load_ext autoreload
%autoreload 2
%reset -f

from devtools import debug  # noqa: F401  # noqa: F811
from dotenv import load_dotenv
from rich import print  # noqa: F401

assert load_dotenv(verbose=True)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
from src.utils.config_mngr import global_config, global_config_reload

global_config_reload()

list_demos = global_config().merge_with("config/schemas/document_extractor.yaml").get_list("Document_extractor_demo")


test_schema = next((item for item in list_demos if item.get("schema_name") == "Rainbow File"))


In [8]:
from src.demos.ekg.struct_rag_doc_processing import StructuredRagConfig, StructuredRagDocProcessor
from src.demos.ekg.struct_rag_tool_factory import StructuredRagToolFactory

KV_STORE = "file"


vector_store_factory = StructuredRagConfig.get_vector_store_factory()
struct_rag_conf = StructuredRagConfig(
    model_definition=test_schema,
    vector_store_factory=vector_store_factory,
    llm_id=None,
    kvstore_id=KV_STORE,
)
rainbow_rag_processor = StructuredRagDocProcessor(rag_conf=struct_rag_conf)
rainbow_tool_factory = StructuredRagToolFactory(rag_conf=struct_rag_conf)


2025-08-20 18:07:49.587 | DEBUG    | src.ai_core.llm_factory:get_llm:588 - get LLM:'kimi_k2_openrouter' -extra: {'temperature': 0.0}


In [9]:
import os
from pathlib import Path

doc_id = "03.RESM-SOL-9000559500_CNES_TMA_VENUS_VIP_PEPS_THEIA_MUSCATE-v0.2"

file1 = Path(os.getenv("ONEDRIVE", "")) / "prj/atos-kg/rainbow-json/" / (doc_id + "_extracted.json")
assert file1.exists()
doc_text = file1.read_text()

rainbow_report = rainbow_rag_processor.analyze_document(
    document_id=doc_id,
    markdown=doc_text,
)

print("Structured result:", rainbow_report)

assert rainbow_report


2025-08-20 18:07:49.653 | DEBUG    | src.utils.pydantic.kv_store:load_object:111 - read 'RainbowProjectAnalysis/03.RESM-SOL-9000559500_CNES_TMA_VENUS_VIP_PEPS_THEIA_MUSCATE-v0.2.json' from KV store
2025-08-20 18:07:49.654 | DEBUG    | src.utils.pydantic.kv_store:load_object:113 - stored_data: {'identification': {'name': 'CNES TMA VENUS VIP_PEPS_ THEIA MUSCATE', 'opportunity_id': '9000559500', 'customer': 'CNES', 'customer_segment': 'Aerospace', 'status': 'Solution Review', 'start_date': '2019-04-01', 'end_date': '2022-03-31'}, 'description': {'objectives': ['Tierce Maintenance Applicative (TMA) of CNES Earth-observation data production centres and their image-processing chains', 'Corrective and evolutionary maintenance of three lots: VENUS VIP, PEPS, THEIA MUSCATE'], 'scope': 'Take-over phase followed by corrective/evolutionary maintenance for three lots (mono-award framework)', 'success_metrics': ['SLA compliance', 'Quality of deliverables', 'Customer satisfaction'], 'differentiators'

Structured result:
RainbowProjectAnalysis(
    identification=ProjectIdentification(
        name='CNES TMA VENUS VIP_PEPS_ THEIA MUSCATE',
        opportunity_id='9000559500',
        customer='CNES',
        customer_segment='Aerospace',
        status='Solution Review',
        start_date='2019-04-01',
        end_date='2022-03-31'
    ),
    description=ProjectDescription(
        objectives=[
            'Tierce Maintenance Applicative (TMA) of CNES Earth-observation data production centres and their 
image-processing chains',
            'Corrective and evolutionary maintenance of three lots: VENUS VIP, PEPS, THEIA MUSCATE'
        ],
        scope='Take-over phase followed by corrective/evolutionary maintenance for three lots (mono-award 
framework)',
        success_metrics=['SLA compliance', 'Quality of deliverables', 'Customer satisfaction'],
        differentiators=[
            'Existing incumbent on PEPS',
            'Deep Earth-observation domain knowledge',
            'MUNDI platform synergies'
        ]
    ),
    team=[
        PersonRole(name='Marc Ferrer', role='Global Client Leader', organization='Atos', contact_type='internal'),
        PersonRole(
            name='Aurore Dorez',
            role='Sales Lead / Deal Maker',
            organization='Atos',
            contact_type='internal'
        ),
        PersonRole(name='Barthélémy Marti', role='Bid Manager', organization='Atos', contact_type='internal'),
        PersonRole(
            name='Olivier Rondeau',
            role='Solution Manager / Contract Executive',
            organization='Atos',
            contact_type='internal'
        ),
        PersonRole(name='Caroline Jaulin', role='Finance Lead', organization='Atos', contact_type='internal'),
        PersonRole(name='Danièle Phankongsy', role='Deal Lawyer', organization='Atos', contact_type='internal'),
        PersonRole(name='Sonia Gouel', role='Project Manager', organization='Atos', contact_type='internal'),
        PersonRole(
            name='Pierre Bourrousse',
            role='Strategy Achat Sponsor',
            organization='CNES',
            contact_type='external'
        ),
        PersonRole(name='Gérard Lassalle-Valler', role='Sponsor', organization='CNES', contact_type='external'),
        PersonRole(
            name='Sylvia Sylvander',
            role='CP CNES Décideur',
            organization='CNES',
            contact_type='external'
        )
    ],
    delivery=DeliveryInfo(
        business_lines=['B1P S BS Toulouse'],
        locations=['Toulouse, France'],
        partners=[
            'ACS (subcontractor for Venus VIP maintenance, imposed by CNES)',
            'PIXSTART (subcontractor for THEIA MUSCATE first-year fixes and knowledge transfer)'
        ],
        technologies=[
            'VENUS VIP image-processing chains',
            'PEPS Sentinel data exploitation platform',
            'THEIA MUSCATE continental-surface data services'
        ]
    ),
    financials=FinancialMetrics(
        tcv=1800000.0,
        annual_revenue=300000.0,
        project_margin=21.0,
        payment_terms='30 days from invoice date, quarterly invoicing'
    ),
    risks=[
        RiskAnalysis(
            risk_description='Penalties due to SLA non-compliance caused by quality or delivery delays',
            impact_level='high',
            mitigation_strategy='Rigorous QA and project management',
            status='open'
        ),
        RiskAnalysis(
            risk_description='Cost overruns from underestimation of rework, missing packages, or non-representative
platforms',
            impact_level='medium',
            mitigation_strategy='Detailed due-diligence and contingency planning',
            status='open'
        ),
        RiskAnalysis(
            risk_description='Competency retention issues due to turnover and declining activity',
            impact_level='medium',
            mitigation_strategy='Knowledge management and retention incentives'

In [10]:
chunks = rainbow_rag_processor.chunck(rainbow_report)
# debug(chunks)


In [11]:
from langchain_core.utils.function_calling import convert_to_openai_tool

dyn_tool = rainbow_tool_factory.create_vector_search_lc_tool()
debug(convert_to_openai_tool(dyn_tool))


/tmp/ipykernel_161152/434192433.py:4 <module>
    convert_to_openai_tool(dyn_tool): {
        'type': 'function',
        'function': {
            'name': 'RainbowProjectAnalysis_retriever',
            'description': (
                '\n'
                "Retrieve information related to documents described as 'Input document for a review meeting (called '"
                "Rainbow') that assess an opportunity for a project and the business proposal, before it's sent to the"
                ' customer..\n'
                "Each document is related to a unique id 'opportunity_id', with is typically a  Unique identifier from"
                ' Salesforce or similar CRM.\n'
                'Argument are:\n'
                '- expanded query.  \n'
                '- selected_sections: a list of section names that best match the query. Select several if you are not'
                ' sure.\n'
                "- entity_keys:  list of 'opportunity_id' mentionned in the context / discussion.

{'type': 'function',
 'function': {'name': 'RainbowProjectAnalysis_retriever',
  'description': "\nRetrieve information related to documents described as 'Input document for a review meeting (called 'Rainbow') that assess an opportunity for a project and the business proposal, before it's sent to the customer..\nEach document is related to a unique id 'opportunity_id', with is typically a  Unique identifier from Salesforce or similar CRM.\nArgument are:\n- expanded query.  \n- selected_sections: a list of section names that best match the query. Select several if you are not sure.\n- entity_keys:  list of 'opportunity_id' mentionned in the context / discussion..\n",
  'parameters': {'properties': {'query': {'description': '\nThe query to the semantic search vector store. \nProvide several variants of the request to improve the semantic matching \n(ex: broaden, examples,...) ',
     'type': 'string'},
    'selected_sections': {'description': "\n                    List of sections relev

In [12]:
r = dyn_tool.invoke({"query": "CNES", "selected_sections": ["team"], "entity_keys": []})
print(r)

2025-08-20 18:07:49.883 | DEBUG    | src.ai_extra.pgvector_factory:create_pg_vector_store:77 - Hybrid search enabled with config: HybridSearchConfig(tsv_column='content_tsv', tsv_lang='pg_catalog.english', fts_query='', fusion_function=<function reciprocal_rank_fusion at 0x7fa6759d1a80>, fusion_function_parameters={}, primary_top_k=4, secondary_top_k=4, index_name='pydantic_fields_qwen3_06b_tsv_index', index_type='GIN')
2025-08-20 18:07:49.929 | DEBUG    | src.ai_extra.pgvector_factory:create_pg_vector_store:93 - Use existing pgvector table : pydantic_fields_qwen3_06b


src/ai_extra/pgvector_factory.py:107 create_pg_vector_store
    vector_store: <langchain_postgres.v2.vectorstores.PGVectorStore object at 0x7fa652f7b1d0> (PGVectorStore)


2025-08-20 18:07:49.968 | DEBUG    | src.ai_extra.pgvector_factory:create_pg_vector_store:116 - Failed to apply hybrid search index: 'PGVectorStore' object has no attribute 'apply_hybrid_search_index'
2025-08-20 18:07:49.969 | DEBUG    | src.ai_core.vector_store_factory:get:228 - get vector store  : PgVector/pydantic_fields_qwen3_06b


# opportunity_id: 9000559500
## team

| name                   | role                   | organization | contact_type |
| ---------------------- | ---------------------- | ------------ | ------------ |
| Danièle Phankongsy     | Deal Lawyer            | Atos         | internal     |
| Sonia Gouel            | Project Manager        | Atos         | internal     |
| Pierre Bourrousse      | Strategy Achat Sponsor | CNES         | external     |
| Gérard Lassalle-Valler | Sponsor                | CNES         | external     |
| Sylvia Sylvander       | CP CNES Décideur       | CNES         | external     |

| name                   | role                   | organization | contact_type |
| ---------------------- | ---------------------- | ------------ | ------------ |
| Danièle Phankongsy     | Deal Lawyer            | Atos         | internal     |
| Sonia Gouel            | Project Manager        | Atos         | internal     |
| Pierre Bourrousse      | Strategy Achat Sponsor | CNES         | external     |
| Gérard Lassalle-Valler | Sponsor                | CNES         | external     |
| Sylvia Sylvander       | CP CNES Décideur       | CNES         | external     |

| name                   | role                   | organization | contact_type |
| ---------------------- | ---------------------- | ------------ | ------------ |
| Danièle Phankongsy     | Deal Lawyer            | Atos         | internal     |
| Sonia Gouel            | Project Manager        | Atos         | internal     |
| Pierre Bourrousse      | Strategy Achat Sponsor | CNES         | external     |
| Gérard Lassalle-Valler | Sponsor                | CNES         | external     |
| Sylvia Sylvander       | CP CNES Décideur       | CNES         | external     |

| name     | role                    | organization   | contact_type |
| -------- | ----------------------- | -------------- | ------------ |
| Abhijeet | Bid Lead / AMS Practice | AMS Practice   | Internal     |
| Anup     | SAP Practice Lead       | SAP Practice   | Internal     |
| Oana     | Legal & Compliance      | Legal          | Internal     |
| Jakub    | Finance                 | Finance        | Internal     |
| E. Mohr  | Customer Management     | Siemens Energy | Customer     |

| name                    | role                    | organization        | contact_type |
| ----------------------- | ----------------------- | ------------------- | ------------ |
| Dominique POILEVEY      | GM COO                  | Solution & Delivery | Expert       |
| Ayşen Öztürk Canbazoğlu | Country Controller      | Finance             | Expert       |
| Bahar YENERER           | General Counsel         | Legal               | Expert       |
| Cengiz Kesoglu          | Head of Risk Management | Risk                | Expert       |
| Yalin Tolga GOKYAR      | Head of Procurement     | Procurement         | Expert       |

| name          | role            | organization | contact_type |
| ------------- | --------------- | ------------ | ------------ |
| Darren Henry  | Bid Lead        | Atos         | Bid Team     |
| Chris         | Bid Team Member | Atos         | Bid Team     |
| Mike Atkinson | Bid Team Member | Atos         | Bid Team     |
| Shalina       | Bid Team Member | Atos         | Bid Team     |
| Srikant P     | Bid Team Member | Atos         | Bid Team     |

| name             | role                                  | organization | contact_type |
| ---------------- | ------------------------------------- | ------------ | ------------ |
| Marc Ferrer      | Global Client Leader                  | Atos         | internal     |
| Aurore Dorez     | Sales Lead / Deal Maker               | Atos         | internal     |
| Barthélémy Marti | Bid Manager                           | Atos         | internal     |
| Olivier Rondeau  | Solution Manager / Contract Executive | Atos         | internal     |
| Caroline Jaulin  | Finance Lead                          | Atos         | internal     |



In [13]:
rainbow_rag_processor.kv_to_vector_store()


2025-08-20 18:07:51.819 | DEBUG    | src.ai_extra.pgvector_factory:create_pg_vector_store:77 - Hybrid search enabled with config: HybridSearchConfig(tsv_column='content_tsv', tsv_lang='pg_catalog.english', fts_query='', fusion_function=<function reciprocal_rank_fusion at 0x7fa6759d1a80>, fusion_function_parameters={}, primary_top_k=4, secondary_top_k=4, index_name='pydantic_fields_qwen3_06b_tsv_index', index_type='GIN')
2025-08-20 18:07:51.859 | DEBUG    | src.ai_extra.pgvector_factory:create_pg_vector_store:93 - Use existing pgvector table : pydantic_fields_qwen3_06b


src/ai_extra/pgvector_factory.py:107 create_pg_vector_store
    vector_store: <langchain_postgres.v2.vectorstores.PGVectorStore object at 0x7fa652fa7fe0> (PGVectorStore)


2025-08-20 18:07:51.887 | DEBUG    | src.ai_extra.pgvector_factory:create_pg_vector_store:116 - Failed to apply hybrid search index: 'PGVectorStore' object has no attribute 'apply_hybrid_search_index'
2025-08-20 18:07:51.888 | DEBUG    | src.ai_core.vector_store_factory:get:228 - get vector store  : PgVector/pydantic_fields_qwen3_06b
2025-08-20 18:07:51.889 | INFO     | src.ai_core.vector_store_factory:delete_collection:319 - drop file public.pydantic_fields_qwen3_06b
2025-08-20 18:07:51.933 | DEBUG    | src.utils.pydantic.kv_store:load_object:111 - read 'RainbowProjectAnalysis/L3-MTN__Group_-_MTN_ecommerce-AFR-9000992602-OFF_Ver2.1__1_.json' from KV store
2025-08-20 18:07:51.934 | DEBUG    | src.utils.pydantic.kv_store:load_object:113 - stored_data: {'identification': {'name': 'eCommerce', 'opportunity_id': '9000992602', 'customer': 'MTN Group', 'customer_segment': 'TMT - Telecom', 'status': 'PUR to HND', 'start_date': '2022-08-18', 'end_date': 'Q4 2022'}, 'description': {'objectives'

src/ai_extra/pgvector_factory.py:107 create_pg_vector_store
    vector_store: <langchain_postgres.v2.vectorstores.PGVectorStore object at 0x7fa652f79730> (PGVectorStore)


2025-08-20 18:07:52.030 | DEBUG    | src.ai_extra.pgvector_factory:create_pg_vector_store:116 - Failed to apply hybrid search index: 'PGVectorStore' object has no attribute 'apply_hybrid_search_index'
2025-08-20 18:07:52.031 | DEBUG    | src.ai_core.vector_store_factory:get:228 - get vector store  : PgVector/pydantic_fields_qwen3_06b
2025-08-20 18:07:52.110 | DEBUG    | src.utils.pydantic.kv_store:load_object:111 - read 'RainbowProjectAnalysis/03.RESM-SOL-9000559500_CNES_TMA_VENUS_VIP_PEPS_THEIA_MUSCATE-v0.2.json' from KV store
2025-08-20 18:07:52.111 | DEBUG    | src.utils.pydantic.kv_store:load_object:113 - stored_data: {'identification': {'name': 'CNES TMA VENUS VIP_PEPS_ THEIA MUSCATE', 'opportunity_id': '9000559500', 'customer': 'CNES', 'customer_segment': 'Aerospace', 'status': 'Solution Review', 'start_date': '2019-04-01', 'end_date': '2022-03-31'}, 'description': {'objectives': ['Tierce Maintenance Applicative (TMA) of CNES Earth-observation data production centres and their im

...

2025-08-20 18:07:52.370 | DEBUG    | src.ai_extra.pgvector_factory:create_pg_vector_store:116 - Failed to apply hybrid search index: 'PGVectorStore' object has no attribute 'apply_hybrid_search_index'
2025-08-20 18:07:52.371 | DEBUG    | src.ai_core.vector_store_factory:get:228 - get vector store  : PgVector/pydantic_fields_qwen3_06b
2025-08-20 18:07:52.458 | DEBUG    | src.utils.pydantic.kv_store:load_object:111 - read 'RainbowProjectAnalysis/RESM-PUR-to-HND_-_9000988694_Cocteau_Extension_-_L1.json' from KV store
2025-08-20 18:07:52.459 | DEBUG    | src.utils.pydantic.kv_store:load_object:113 - stored_data: {'identification': {'name': 'Cocteau Contract Extension', 'opportunity_id': '9001022639 & 9001046954', 'customer': 'Standard Chartered Bank', 'customer_segment': 'Banking', 'status': 'Contract Rainbow', 'start_date': '2024-01-01', 'end_date': '2026-06-30'}, 'description': {'objectives': ['Extend the existing Cocteau contract for 2.5 years (Jan 2024 – Jun 2026) to bridge until Zeus 

src/ai_extra/pgvector_factory.py:107 create_pg_vector_store
    vector_store: <langchain_postgres.v2.vectorstores.PGVectorStore object at 0x7fa6531aac90> (PGVectorStore)


2025-08-20 18:07:52.550 | DEBUG    | src.ai_extra.pgvector_factory:create_pg_vector_store:116 - Failed to apply hybrid search index: 'PGVectorStore' object has no attribute 'apply_hybrid_search_index'
2025-08-20 18:07:52.550 | DEBUG    | src.ai_core.vector_store_factory:get:228 - get vector store  : PgVector/pydantic_fields_qwen3_06b
2025-08-20 18:07:52.633 | DEBUG    | src.utils.pydantic.kv_store:load_object:111 - read 'RainbowProjectAnalysis/RESM-FT_-_9001032923__Dominos_Pizza_Service_Desk_and_Field_Support_20221117V01_L5.json' from KV store
2025-08-20 18:07:52.634 | DEBUG    | src.utils.pydantic.kv_store:load_object:113 - stored_data: {'identification': {'name': "Domino's Pizza Service Desk and Field Support", 'opportunity_id': '9001032923', 'customer': 'Dominos Pizza', 'customer_segment': 'Commercial', 'status': 'Strategy-Contract', 'start_date': '2022-11-17', 'end_date': None, 'region': 'GM - Growing Markets', 'country': 'Turkey', 'business_line': 'Digital Workplace', 'lead_cc': '

src/ai_extra/pgvector_factory.py:107 create_pg_vector_store
    vector_store: <langchain_postgres.v2.vectorstores.PGVectorStore object at 0x7fa65305fbf0> (PGVectorStore)


2025-08-20 18:07:52.727 | DEBUG    | src.ai_extra.pgvector_factory:create_pg_vector_store:116 - Failed to apply hybrid search index: 'PGVectorStore' object has no attribute 'apply_hybrid_search_index'
2025-08-20 18:07:52.728 | DEBUG    | src.ai_core.vector_store_factory:get:228 - get vector store  : PgVector/pydantic_fields_qwen3_06b


In [14]:
# 2. Index the document
rainbow_rag_processor.store_chunks(chunks)
print("Document stored.")


2025-08-20 18:07:52.873 | DEBUG    | src.ai_extra.pgvector_factory:create_pg_vector_store:77 - Hybrid search enabled with config: HybridSearchConfig(tsv_column='content_tsv', tsv_lang='pg_catalog.english', fts_query='', fusion_function=<function reciprocal_rank_fusion at 0x7fa6759d1a80>, fusion_function_parameters={}, primary_top_k=4, secondary_top_k=4, index_name='pydantic_fields_qwen3_06b_tsv_index', index_type='GIN')
2025-08-20 18:07:52.920 | DEBUG    | src.ai_extra.pgvector_factory:create_pg_vector_store:93 - Use existing pgvector table : pydantic_fields_qwen3_06b


src/ai_extra/pgvector_factory.py:107 create_pg_vector_store
    vector_store: <langchain_postgres.v2.vectorstores.PGVectorStore object at 0x7fa65232e570> (PGVectorStore)


2025-08-20 18:07:52.945 | DEBUG    | src.ai_extra.pgvector_factory:create_pg_vector_store:116 - Failed to apply hybrid search index: 'PGVectorStore' object has no attribute 'apply_hybrid_search_index'
2025-08-20 18:07:52.946 | DEBUG    | src.ai_core.vector_store_factory:get:228 - get vector store  : PgVector/pydantic_fields_qwen3_06b


Document stored.

In [15]:
hits = rainbow_tool_factory.query_vectorstore("e-mail address", k=2)
print("Vector hits:", hits)

2025-08-20 18:07:53.097 | DEBUG    | src.ai_extra.pgvector_factory:create_pg_vector_store:77 - Hybrid search enabled with config: HybridSearchConfig(tsv_column='content_tsv', tsv_lang='pg_catalog.english', fts_query='', fusion_function=<function reciprocal_rank_fusion at 0x7fa6759d1a80>, fusion_function_parameters={}, primary_top_k=4, secondary_top_k=4, index_name='pydantic_fields_qwen3_06b_tsv_index', index_type='GIN')
2025-08-20 18:07:53.146 | DEBUG    | src.ai_extra.pgvector_factory:create_pg_vector_store:93 - Use existing pgvector table : pydantic_fields_qwen3_06b


src/ai_extra/pgvector_factory.py:107 create_pg_vector_store
    vector_store: <langchain_postgres.v2.vectorstores.PGVectorStore object at 0x7fa652547ce0> (PGVectorStore)


2025-08-20 18:07:53.177 | DEBUG    | src.ai_extra.pgvector_factory:create_pg_vector_store:116 - Failed to apply hybrid search index: 'PGVectorStore' object has no attribute 'apply_hybrid_search_index'
2025-08-20 18:07:53.178 | DEBUG    | src.ai_core.vector_store_factory:get:228 - get vector store  : PgVector/pydantic_fields_qwen3_06b


Vector hits:
[
    Document(
        id='d405a1a4-27b3-4040-9bd5-798d63034d35',
        metadata={
            'description': 'Financial metrics',
            'document_id': 'RESM-FT_-_9001032923__Dominos_Pizza_Service_Desk_and_Field_Support_20221117V01_L5',
            'entity_id': '9001032923',
            'field_name': 'financials',
            'model_class': 'RainbowProjectAnalysis'
        },
        page_content='# tcv\n918796.0\n# annual_revenue\n304523.0\n# project_margin\n0.23\n# payment_terms\n30 
days\n# currency\nEUR\n# inflation_clause\nTrue\n# cash_positive_month\n12'
    ),
    Document(
        id='b7cd5393-4970-451b-aa35-f8b10814a99d',
        metadata={
            'description': 'Project identification information',
            'document_id': 'L3-MTN__Group_-_MTN_ecommerce-AFR-9000992602-OFF_Ver2.1_(1)',
            'entity_id': '9000992602',
            'field_name': 'identification',
            'model_class': 'RainbowProjectAnalysis'
        },
        page_content='# name\neCommerce\n# opportunity_id\n9000992602\n# customer\nMTN Group\n# 
customer_segment\nTMT - Telecom\n# status\nPUR to HND\n# start_date\n2022-08-18\n# end_date\nQ4 2022'
    )
]

In [16]:
# 3. Query the vector store
hits = rainbow_tool_factory.query_vectorstore("revenue", k=2, filter={"field_name": {"$eq": "financials"}})
print("Vector hits:", hits)

2025-08-20 18:07:54.193 | DEBUG    | src.ai_extra.pgvector_factory:create_pg_vector_store:77 - Hybrid search enabled with config: HybridSearchConfig(tsv_column='content_tsv', tsv_lang='pg_catalog.english', fts_query='', fusion_function=<function reciprocal_rank_fusion at 0x7fa6759d1a80>, fusion_function_parameters={}, primary_top_k=4, secondary_top_k=4, index_name='pydantic_fields_qwen3_06b_tsv_index', index_type='GIN')
2025-08-20 18:07:54.226 | DEBUG    | src.ai_extra.pgvector_factory:create_pg_vector_store:93 - Use existing pgvector table : pydantic_fields_qwen3_06b


src/ai_extra/pgvector_factory.py:107 create_pg_vector_store
    vector_store: <langchain_postgres.v2.vectorstores.PGVectorStore object at 0x7fa652290380> (PGVectorStore)


2025-08-20 18:07:54.252 | DEBUG    | src.ai_extra.pgvector_factory:create_pg_vector_store:116 - Failed to apply hybrid search index: 'PGVectorStore' object has no attribute 'apply_hybrid_search_index'
2025-08-20 18:07:54.252 | DEBUG    | src.ai_core.vector_store_factory:get:228 - get vector store  : PgVector/pydantic_fields_qwen3_06b


Vector hits:
[
    Document(
        id='19201ceb-1d8e-435a-bc60-e3a03e77673b',
        metadata={
            'description': 'Financial metrics',
            'document_id': 'RESM-PUR-to-HND_-_9001096997_-_L2',
            'entity_id': '9001096997',
            'field_name': 'financials',
            'model_class': 'RainbowProjectAnalysis'
        },
        page_content="# tcv\n106900000.0\n# annual_revenue\n21380000.0\n# project_margin\n0.347\n# 
payment_terms\nMonthly invoicing in EUR\n# pricing_model\n\n | AMS                       | SAP                     
| SFDC                    | Transition                | Governance              |\n | ------------------------- | 
------------------------- | ----------------------- | ------------------------- | ----------------------- |\n | 
{'T&M': 0.98, 'FP': 0.02} | {'T&M': 0.32, 'FP': 0.68} | {'T&M': 1.0, 'FP': 0.0} | {'T&M': 0.86, 'FP': 0.14} | 
{'T&M': 0.0, 'FP': 1.0} |\n\n# inflation\n\n | India                                    | Germany                  
| Poland                                   | Mexico                                  | USA                         
| China                          |\n | ---------------------------------------- | 
---------------------------------------- | ---------------------------------------- | 
--------------------------------------- | ------------------------------ | ------------------------------ |\n | 
[0.0667, 0.0334, 0.0334, 0.0334, 0.0334] | [0.0866, 0.0433, 0.0433, 0.0433, 0.0433] | [0.1436, 0.0718, 0.0718, 
0.0718, 0.0718] | [0.079, 0.0395, 0.0395, 0.0395, 0.0395] | [0.08, 0.04, 0.04, 0.04, 0.04] | [0.02, 0.01, 0.01, 
0.01, 0.01] |\n\n# risk_budget_percent\n4.5"
    ),
    Document(
        id='d405a1a4-27b3-4040-9bd5-798d63034d35',
        metadata={
            'description': 'Financial metrics',
            'document_id': 'RESM-FT_-_9001032923__Dominos_Pizza_Service_Desk_and_Field_Support_20221117V01_L5',
            'entity_id': '9001032923',
            'field_name': 'financials',
            'model_class': 'RainbowProjectAnalysis'
        },
        page_content='# tcv\n918796.0\n# annual_revenue\n304523.0\n# project_margin\n0.23\n# payment_terms\n30 
days\n# currency\nEUR\n# inflation_clause\nTrue\n# cash_positive_month\n12'
    )
]

In [17]:
# vector_store_factory.delete_collection()

In [18]:
from smolagents import CodeAgent, LiteLLMModel, Tool

from src.ai_core.llm_factory import LlmFactory

MODEL_ID = None
llm_factory = LlmFactory(llm_id=MODEL_ID, llm_params={"temperature": 0.7})
llm = LiteLLMModel(model_id=llm_factory.get_litellm_model_name(), **llm_factory.llm_params)

dyn_tool = rainbow_tool_factory.create_vector_search_lc_tool()
sa_tool = Tool.from_langchain(dyn_tool)

agent = CodeAgent(tools=[sa_tool], model=llm)

agent.run("What are the offerings in space sector ?")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What are the offerings in space sector ?                                                                        │
│                                                                                                                 │
╰─ LiteLLMModel - openrouter/moonshotai/kimi-k2 ──────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

/home/tcl/prj/genai-blueprint/.venv/lib/python3.12/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 9 fields but got 5: Expected `Message` - serialized value may not be as expected [input_value=Message(content='Thought:...one, 'reasoning': None}), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [input_value=Choices(finish_reason='st...finish_reason': 'stop'}), input_type=Choices])
  return self.__pydantic_serializer__.to_python(


─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  space_offerings = rainbowprojectanalysis_retriever(                                                              
      query="space sector offerings products services capabilities solutions",                                     
      selected_sections=["offerings", "products", "services", "capabilities", "solutions"],                        
      entity_keys=[]                                                                                               
  )                                                                                                                
  print(space_offerings)                                                                                           
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

2025-08-20 18:08:01.875 | WARNING  | src.demos.ekg.struct_rag_tool_factory:_run:90 - Removing invalid section: ['offerings', 'products', 'services', 'capabilities', 'solutions']
2025-08-20 18:08:01.906 | DEBUG    | src.ai_extra.pgvector_factory:create_pg_vector_store:77 - Hybrid search enabled with config: HybridSearchConfig(tsv_column='content_tsv', tsv_lang='pg_catalog.english', fts_query='', fusion_function=<function reciprocal_rank_fusion at 0x7fa6759d1a80>, fusion_function_parameters={}, primary_top_k=4, secondary_top_k=4, index_name='pydantic_fields_qwen3_06b_tsv_index', index_type='GIN')
2025-08-20 18:08:02.072 | DEBUG    | src.ai_extra.pgvector_factory:create_pg_vector_store:93 - Use existing pgvector table : pydantic_fields_qwen3_06b


...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 4 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  space_offerings_summary = {                                                                                      
      "primary_space_sector_offering": "Earth-observation data processing and management services",                
      "specific_services": [                                                                                       
          "Corrective and evolutionary maintenance of Earth-observation data production centers",                  
          "Image processing chain maintenance and enhancement",                                                    
          "Tierce Maintenance Applicative (TMA) for space-based data systems"                                      
      ],                                                                                                           
      "key_platforms": [                                                                                           
          "VENUS VIP - satellite data processing",                                                                 
          "PEPS - Earth-observation data platform",                                                                
          "THEIA MUSCATE - land surface observation processing chain"                                              
      ],                                                                                                           
      "capabilities": [                                                                                            
          "Deep Earth-observation domain expertise",                                                               
          "Satellite data processing and analysis",                                                                
          "Space-based data platform maintenance",                                                                 
          "Synergies with MUNDI platform (Multi-mission Earth Observation)"                                        
      ],                                                                                                           
      "customer_segment": "Aerospace and Space Agencies",                                                          
      "service_type": "Application maintenance and support services for space systems"                             
  }                                                                                                                
                                                                                                                   
  final_answer(space_offerings_summary)                                                                            
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Out - Final answer: {'primary_space_sector_offering': 'Earth-observation data processing and management services', 
'specific_services': ['Corrective and evolutionary maintenance of Earth-observation data production centers', 
'Image processing chain maintenance and enhancement', 'Tierce Maintenance Applicative (TMA) for space-based data 
systems'], 'key_platforms': ['VENUS VIP - satellite data processing', 'PEPS - Earth-observation data platform', 
'THEIA MUSCATE - land surface observation processing chain'], 'capabilities': ['Deep Earth-observation domain 
expertise', 'Satellite data processing and analysis', 'Space-based data platform maintenance', 'Synergies with 
MUNDI platform (Multi-mission Earth Observation)'], 'customer_segment': 'Aerospace and Space Agencies', 
'service_type': 'Application maintenance and support services for space systems'}

[Step 4: Duration 9.61 seconds| Input tokens: 16,206 | Output tokens: 587]

{'primary_space_sector_offering': 'Earth-observation data processing and management services',
 'specific_services': ['Corrective and evolutionary maintenance of Earth-observation data production centers',
  'Image processing chain maintenance and enhancement',
  'Tierce Maintenance Applicative (TMA) for space-based data systems'],
 'key_platforms': ['VENUS VIP - satellite data processing',
  'PEPS - Earth-observation data platform',
  'THEIA MUSCATE - land surface observation processing chain'],
 'capabilities': ['Deep Earth-observation domain expertise',
  'Satellite data processing and analysis',
  'Space-based data platform maintenance',
  'Synergies with MUNDI platform (Multi-mission Earth Observation)'],
 'customer_segment': 'Aerospace and Space Agencies',
 'service_type': 'Application maintenance and support services for space systems'}